# 07 Benchmark CPU

Cel: zmierzyc latency p50/p95 i throughput dla Teacher FP32, Student FP32 i Student INT8.


In [ ]:
# Krok 1: importy i konfiguracja benchmarku
# Po co to robimy: przygotowujemy biblioteki i stale, aby porownac szybkosc i przepustowosc modeli na CPU.

# Importujemy json do zapisu wynikow benchmarku.
import json
# Importujemy os do pracy ze sciezkami i katalogami.
import os
# Importujemy time do pomiaru czasu inferencji.
import time
# Importujemy numpy do liczenia percentyli p50 i p95.
import numpy as np
# Importujemy PyTorch do inferencji modelu.
import torch
# Importujemy loader danych z dysku.
from datasets import load_from_disk
# Importujemy klase modelu klasyfikacyjnego z Transformers.
from transformers import AutoModelForSequenceClassification

# Ustalamy urzadzenie benchmarku na CPU.
DEVICE = torch.device("cpu")
# Ustalamy liczbe krokow warmup przed pomiarem.
WARMUP_STEPS = 50
# Ustalamy liczbe krokow pomiarowych.
MEASURE_STEPS = 200
# Ustalamy rozmiary batchy do porownania (online i mini-batch).
BATCHES = [1, 8]


In [ ]:
# Krok 2: helper do benchmarku pojedynczego modelu
# Po co to robimy: jedna funkcja mierzy p50, p95 i throughput dla dowolnego modelu i batcha.

# Definiujemy funkcje benchmarku dla jednego modelu.
def run_benchmark(model, input_ids, attention_mask, warmup_steps, measure_steps):
    # Ustawiamy model w tryb ewaluacji.
    model.eval()
    # Przenosimy model na urzadzenie CPU.
    model.to(DEVICE)

    # Wykonujemy warmup, aby ustabilizowac pomiar czasu.
    with torch.no_grad():
        # Iterujemy przez ustalona liczbe krokow warmup.
        for _ in range(warmup_steps):
            # Wykonujemy inferencje bez zapisu wyniku.
            _ = model(input_ids=input_ids, attention_mask=attention_mask).logits

    # Tworzymy liste na czasy pojedynczych inferencji w ms.
    times_ms = []
    # Wykonujemy wlasciwy pomiar benchmarku.
    with torch.no_grad():
        # Iterujemy przez ustalona liczbe krokow pomiarowych.
        for _ in range(measure_steps):
            # Zapisujemy czas startu inferencji.
            t0 = time.perf_counter()
            # Wykonujemy inferencje modelu.
            _ = model(input_ids=input_ids, attention_mask=attention_mask).logits
            # Zapisujemy czas konca inferencji.
            t1 = time.perf_counter()
            # Przeliczamy czas na milisekundy i dopisujemy do listy.
            times_ms.append((t1 - t0) * 1000.0)

    # Liczymy mediane czasu (p50).
    p50 = float(np.percentile(times_ms, 50))
    # Liczymy wysoki percentyl opoznienia (p95).
    p95 = float(np.percentile(times_ms, 95))
    # Liczymy throughput jako liczbe rekordow na sekunde.
    throughput = float((input_ids.shape[0] * measure_steps) / (sum(times_ms) / 1000.0))
    # Zwracamy slownik metryk benchmarku.
    return {"p50_ms": p50, "p95_ms": p95, "throughput_req_s": throughput}


In [ ]:
# Krok 3: przygotowanie probki wejsciowej z tokenized danych
# Po co to robimy: benchmarkujemy modele na tym samym wejsciu, aby porownanie bylo uczciwe.

# Wczytujemy tokenized dane studenta z dysku.
student_ds = load_from_disk("../artifacts/tokenized_student")
# Pobieramy jedna probe testowa jako wzorzec wejscia.
sample = student_ds["test"][0]

# Definiujemy funkcje, ktora buduje batch przez powielenie tej samej probki.
def make_batch(batch_size):
    # Budujemy input_ids o zadanym batch_size.
    input_ids = sample["input_ids"].unsqueeze(0).repeat(batch_size, 1).to(DEVICE)
    # Budujemy attention_mask o zadanym batch_size.
    attention_mask = sample["attention_mask"].unsqueeze(0).repeat(batch_size, 1).to(DEVICE)
    # Zwracamy przygotowane tensory wejscia.
    return input_ids, attention_mask


In [ ]:
# Krok 4: ladowanie modeli
# Po co to robimy: przygotowujemy trzy warianty do porownania: teacher FP32, student FP32 i student INT8.

# Wybieramy katalog teachera: preferujemy wersje finetuned, fallback do starszej wersji.
teacher_path = "../artifacts/teacher_finetuned"
# Jesli finetuned teacher nie istnieje, przechodzimy na starszy katalog teacher.
if not os.path.exists(os.path.join(teacher_path, "config.json")):
    # Ustawiamy fallback na katalog podstawowy teachera.
    teacher_path = "../artifacts/teacher"
# Wczytujemy model teachera z wybranego katalogu.
teacher = AutoModelForSequenceClassification.from_pretrained(teacher_path)

# Ustalamy domyslny katalog studenta FP32 jako distilled.
student_fp32_path = "../artifacts/student_fp32_distilled"
# Jesli distilled nie istnieje, przechodzimy na baseline.
if not os.path.exists(os.path.join(student_fp32_path, "config.json")):
    # Ustawiamy fallback na katalog baseline studenta.
    student_fp32_path = "../artifacts/student_fp32_baseline"
# Wczytujemy model studenta FP32.
student_fp32 = AutoModelForSequenceClassification.from_pretrained(student_fp32_path)

# Budujemy model INT8 z modelu FP32, aby uniknac problemu torch.load w PyTorch 2.6.
student_int8 = torch.quantization.quantize_dynamic(
    # Podajemy model FP32 jako zrodlo quantization.
    student_fp32,
    # Quantyzujemy warstwy Linear.
    {torch.nn.Linear},
    # Ustawiamy typ wag na qint8.
    dtype=torch.qint8,
)

# Wczytujemy zapisany state_dict INT8 (bez pelnego unpicklingu obiektu).
int8_state = torch.load("../artifacts/student_int8/model_int8_state_dict.pt", map_location="cpu")
# Ladowanie wag INT8 do zbudowanego modelu quantized.
student_int8.load_state_dict(int8_state)


In [ ]:
# Krok 5: uruchomienie benchmarku
# Po co to robimy: liczymy metryki wydajnosci dla wszystkich modeli i batchy, a potem zapisujemy wyniki do JSON.

# Tworzymy pusty slownik na wyniki benchmarku.
results = {}
# Iterujemy po zadanych rozmiarach batcha.
for bs in BATCHES:
    # Budujemy wejscie dla aktualnego batch_size.
    input_ids, attention_mask = make_batch(bs)
    # Mierzymy benchmark teachera FP32.
    results[f"teacher_fp32_bs{bs}"] = run_benchmark(teacher, input_ids, attention_mask, WARMUP_STEPS, MEASURE_STEPS)
    # Mierzymy benchmark studenta FP32.
    results[f"student_fp32_bs{bs}"] = run_benchmark(student_fp32, input_ids, attention_mask, WARMUP_STEPS, MEASURE_STEPS)
    # Mierzymy benchmark studenta INT8.
    results[f"student_int8_bs{bs}"] = run_benchmark(student_int8, input_ids, attention_mask, WARMUP_STEPS, MEASURE_STEPS)

# Tworzymy katalog outputs, jesli jeszcze nie istnieje.
os.makedirs("../outputs", exist_ok=True)
# Otwieramy plik JSON do zapisu wynikow benchmarku.
with open("../outputs/bench_results.json", "w", encoding="utf-8") as f:
    # Zapisujemy wyniki benchmarku w czytelnym formacie.
    json.dump(results, f, indent=2)

# Wyswietlamy wyniki benchmarku w konsoli notebooka.
print(json.dumps(results, indent=2))
